# Legal AI Agent — Serve API trên Colab (demo + test models)

**Mục tiêu:** chạy toàn bộ RAG API trên Colab, expose qua tunnel public —
**máy bạn không cần tải 56GB về**, UI local chỉ cần trỏ vào URL.

> ⚙️ **Runtime bắt buộc: GPU T4 + bật "RAM cao cấp" (High-RAM)**
> (HNSW index 21GB load vào RAM — máy chuẩn 12GB sẽ crash).
> T4 High-RAM tốn ~2–3 units/giờ.

| Bước | Thời gian |
|---|---|
| Cài đặt + clone | ~5 phút |
| Tải vectorstore từ HF | ~10–15 phút |
| Khởi động API + tunnel | ~2 phút |
| Câu hỏi đầu tiên (load model + index) | ~3–5 phút |
| **Tổng tới lúc demo được** | **~25–30 phút** |

**Secrets cần có (🔑 sidebar):** `HF_TOKEN` (tải store private), `KIEAI_API_KEY` (LLM).
Tùy chọn: `GEMINI_API_KEY`, `GROQ_API_KEY`.

In [ ]:
# ── Cell 1: Kiểm tra môi trường ──────────────────────────────────────────────
try:
    import google.colab  # noqa
except ImportError:
    raise SystemExit("Notebook này chỉ chạy trên Google Colab.")

import psutil, shutil, torch

ram_gb = psutil.virtual_memory().total / 1e9
disk_free = shutil.disk_usage("/content").free / 1e9
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ"

print(f"GPU : {gpu}")
print(f"RAM : {ram_gb:.0f} GB  {'✓' if ram_gb > 40 else '❌ CẦN High-RAM! Runtime → Change runtime type → bật RAM cao cấp'}")
print(f"Disk: {disk_free:.0f} GB free  {'✓' if disk_free > 55 else '❌ cần ≥55GB'}")

if ram_gb < 40:
    raise SystemExit("RAM không đủ để load HNSW 21GB — bật High-RAM rồi chạy lại.")


In [ ]:
# ── Cell 2: Clone repo + cài dependencies + cloudflared ─────────────────────
import os
os.chdir("/content")

if os.path.exists("/content/ProjectGenAI_2/.git"):
    os.chdir("/content/ProjectGenAI_2")
    !git pull origin main
else:
    !git clone -q https://github.com/HoangNhatTR/ProjectGenAI_2.git
    os.chdir("/content/ProjectGenAI_2")

print("Installing dependencies (~3-5 phút)...")
!pip install -q -r requirements.txt

# cloudflared — quick tunnel, không cần tài khoản
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
print("✓ Done")


In [ ]:
# ── Cell 3: Cấu hình .env (đọc keys từ Colab Secrets) ───────────────────────
import os, secrets, time
from google.colab import userdata

def _sec(name):
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""

KIEAI_API_KEY = _sec("KIEAI_API_KEY")
HF_TOKEN      = _sec("HF_TOKEN")
GEMINI_API_KEY = _sec("GEMINI_API_KEY")
GROQ_API_KEY   = _sec("GROQ_API_KEY")

assert HF_TOKEN, "Thiếu HF_TOKEN trong Colab Secrets (cần để tải vectorstore private)"
assert KIEAI_API_KEY, "Thiếu KIEAI_API_KEY trong Colab Secrets (cần cho LLM)"

# API key bảo vệ endpoint public — nhập vào UI ở mục Settings → Kết nối
API_AUTH_KEY = secrets.token_hex(8)

env = f"""# Auto-generated (Colab serve) — {time.strftime('%Y-%m-%d %H:%M')}
LLM_PROVIDER=kieai
LLM_MODEL=deepseek-chat
ROUTER_MODEL=deepseek-chat
KIEAI_API_KEY={KIEAI_API_KEY}
KIEAI_BASE_URL=https://kieai.erweima.ai/api/v1
GEMINI_API_KEY={GEMINI_API_KEY}
GROQ_API_KEY={GROQ_API_KEY}
HF_TOKEN={HF_TOKEN}

EMBEDDING_MODEL=BAAI/bge-m3
VECTORSTORE_DIR=/content/store/chroma
COLLECTION_NAME=legal_docs
PARENT_STORE_PATH=/content/store/parent_store.db
USE_PARENT_CHILD=true
USE_HYDE=false
TOP_K=5

# Public tunnel → bắt buộc auth + CORS mở (credentials off)
API_AUTH_KEY={API_AUTH_KEY}
CORS_ORIGINS=*
"""
with open("/content/ProjectGenAI_2/.env", "w") as f:
    f.write(env)

print("✓ .env created")
print(f"🔑 API KEY cho UI: {API_AUTH_KEY}   ← nhập vào Settings → Kết nối → API Key")


In [ ]:
# ── Cell 4: Tải vectorstore từ HF (~47GB, 10–15 phút) ───────────────────────
import time
from huggingface_hub import snapshot_download

t0 = time.time()
snapshot_download(
    repo_id="HoangNhat1304/legalai-vectorstore",
    repo_type="dataset",
    local_dir="/content/store",
    allow_patterns=["chroma/**", "parent_store.db"],  # KHÔNG tải bm25 9.2GB (xem ghi chú)
    token=HF_TOKEN,
)
print(f"✓ Store sẵn sàng sau {(time.time()-t0)/60:.1f} phút")
!du -sh /content/store/*

# Ghi chú: BM25 index 9.2GB cần ~25GB RAM khi load bằng rank_bm25 → bỏ qua.
# Retriever tự degrade về vector + CrossEncoder rerank — chất lượng vẫn tốt.


In [ ]:
# ── Cell 5: Khởi động API server (nền) ───────────────────────────────────────
import os, subprocess, time

os.chdir("/content/ProjectGenAI_2")
log = open("/content/api.log", "w")
proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT, cwd="/content/ProjectGenAI_2",
)
print(f"API đang khởi động (pid={proc.pid})...")

deadline = time.time() + 900
ready = False
while time.time() < deadline:
    txt = open("/content/api.log").read()
    if "Sẵn sàng" in txt or "Application startup complete" in txt:
        ready = True
        break
    if proc.poll() is not None:
        break
    time.sleep(5)

print(open("/content/api.log").read()[-1500:])
print("✓ API READY" if ready else "❌ API chưa sẵn sàng — xem log trên")


In [ ]:
# ── Cell 6: Mở tunnel public (cloudflared) ───────────────────────────────────
import re, subprocess, time

tlog = open("/content/tunnel.log", "w")
tproc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=tlog, stderr=subprocess.STDOUT,
)

url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break

assert url, "Không lấy được URL tunnel — xem /content/tunnel.log"
API_URL = url
print("=" * 60)
print(f"🌐 API URL : {API_URL}")
print(f"🔑 API Key : {API_AUTH_KEY}")
print("=" * 60)
print("Nhập 2 giá trị này vào UI local: Settings → tab Kết nối")


In [ ]:
# ── Cell 7: Smoke test (câu đầu chậm 3–5 phút: load BGE-M3 + HNSW 21GB) ─────
import requests, time

q = "xe máy vượt đèn đỏ bị phạt bao nhiêu tiền?"
print(f"Hỏi: {q}\n⏳ Câu đầu tiên load model + index — kiên nhẫn...")

t0 = time.time()
r = requests.post(
    f"{API_URL}/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_AUTH_KEY}"},
    json={"model": "legal-ai", "messages": [{"role": "user", "content": q}], "stream": False},
    timeout=900,
)
print(f"\n⏱ {time.time()-t0:.0f}s — HTTP {r.status_code}\n")
print(r.json()["choices"][0]["message"]["content"][:1200])


In [ ]:
# ── Cell 8: So sánh nhiều model AI trên cùng 1 câu hỏi ──────────────────────
import requests, time

MODELS = ["deepseek-chat", "gpt-4o-mini", "claude-haiku", "gemini-2.0-flash"]
q = "không đội mũ bảo hiểm khi đi xe máy bị phạt bao nhiêu?"

print(f"Câu hỏi: {q}\n")
for m in MODELS:
    t0 = time.time()
    try:
        r = requests.post(
            f"{API_URL}/v1/chat/completions",
            headers={"Authorization": f"Bearer {API_AUTH_KEY}"},
            json={"model": "legal-ai",
                  "messages": [{"role": "user", "content": q}],
                  "stream": False, "llm_model": m},
            timeout=300,
        )
        ans = r.json()["choices"][0]["message"]["content"]
        first = ans.strip().splitlines()[0][:100]
        print(f"  {m:<20} {time.time()-t0:5.1f}s | {first}")
    except Exception as e:
        print(f"  {m:20} LỖI: {str(e)[:80]}")


## Kết nối UI local (legal-chat-ui)

Trên máy bạn:
```powershell
cd C:\Users\HP\Desktop\HoangNhatpr\Project2\legal-chat-ui
npm run dev    # → http://localhost:3001
```

Trong UI → **Settings → tab Kết nối**:
- **API URL** = URL `https://....trycloudflare.com` in ở Cell 6 (không có `/` cuối)
- **API Key** = key in ở Cell 3/6

**Test model khác nhau**: Settings → tab Model → chọn LLM (DeepSeek/GPT-4o/Claude/Gemini
qua KieAI). ⚠ Các model `cc/`, `gh/` (Router9) KHÔNG chạy được từ Colab — Router9 là
proxy localhost trên máy bạn.

**Lưu ý vận hành:**
- Giữ tab Colab mở trong suốt buổi demo (đóng tab → tunnel + API chết)
- URL trycloudflare đổi mỗi lần chạy lại Cell 6 — cập nhật lại vào UI
- Xong demo: Runtime → Disconnect and delete runtime (tiết kiệm units)